# Prepare LLM with logprobs

In [ ]:
import requests
import json

message = "Hello, how are you today?"
url = ""
key = ""
payload = json.dumps({
    "model": "gpt-4o-mini",  
    "messages": [{"role": "user", "content": message}],
    "max_tokens": 1,
    "logprobs": True,
    "top_logprobs": 10
})
headers = {
    'Authorization': f'Bearer {key}',
    'Content-Type': 'application/json'
}
response = requests.request("POST", url, headers=headers, data=payload)
response = json.loads(response.text)

response = response['choices'][0]['message']['content']
response



# Main Experiment

In [ ]:
from datasets import load_dataset
from truthfulrag import TruthfulRAG

# Load dataset
dataset_name = 'timeqa_2022_nota' # musique_negative, squad_negative, timeqa_2022_nota
dataset = load_dataset("json", data_files=f"./datas/{dataset_name}.json")
dataset = dataset['train']



In [ ]:
# Initialize TruthfulRAG pipeline
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_BASE_URL"] = ""

rag = TruthfulRAG(
    dataset=dataset,
    backend_type="hf",  # or "hf", "ollama", "openai"
    model_name="Qwen2.5-7B-Instruct", # Mistral-7B-Instruct-v0.3, gpt-4o-mini
    similarity_model="all-MiniLM-L6-v2",
    output_dir="./results",
    working_dir="./kg_cache",
    threshold=3
)

In [ ]:
import asyncio

async def run_pipeline():
    elements_list = []
    for item in dataset:
        # Generate KG
        await rag.make_knowledge_graph(item)
    
        # Retrieve related entities and relationships
        elements = await rag.knowledge_graph_retrieve(item)
        print(f"elements: {elements}\n\n")
        filtered_elements = await rag.entropy_based_filter(sample=item, elements=elements)
        print(f"filtered_elements: {filtered_elements}\n\n")
        elements_list.append({"id": item['id'], "element": filtered_elements})
    
    # Generate predictions
    predictions = await rag.get_predictions(
        dataset, 
        elements_list,
        generation_type="cot"
    )
    print("predictions: ", predictions)
    
    # Evaluate results
    results = rag.evaluate(dataset, predictions, cot_format=True)
    print("Evaluation Results:")
    print(f"Exact Match: {results['exact_match']:.2f}%")
    print(f"Accuracy: {results['acc']:.2f}%")
    print(f"F1 Score: {results['f1']:.2f}%")

try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    asyncio.run(run_pipeline())
else:
    await run_pipeline()

# calculate average logits (experiment 3)

In [ ]:
import json
from datasets import load_dataset
from truthfulrag import TruthfulRAG

# Load dataset
dataset_name = 'musique_negative'
dataset = load_dataset("json", data_files=f"./datas/{dataset_name}.json")
dataset = dataset['train']
dataset

In [ ]:
# kg_elements
ids = ""
with open(f"./results/{ids}/elements_list.json", "r") as f:
    elements_list = json.load(f)


In [ ]:
import json
import requests
import numpy as np
from truthfulrag.prompts import PromptGenerator
from truthfulrag.utils import (
    encode_string_by_tiktoken,
    decode_tokens_by_tiktoken
)

key = ""
url = ""
backend_type = "openai"
prompt_generator = PromptGenerator(
    llm_type=backend_type,
    task="qa"
)

def get_avg_logprob(prompt_generator, item, elements=None):
    if elements:
        baseline_prompt = prompt_generator.generate_qa_prompt(
            context=item.get('context', ''),
            question=item['question'],
            options=item.get('choices'),
            facts=elements
        )
    else:
        baseline_prompt = prompt_generator.generate_qa_prompt(
            context=item.get('context', ''),
            question=item['question'],
            options=item.get('choices'),
            facts=""
        )
    messages = []
    messages.append({"role": "system", "content": prompt_generator.system_prompt})
    messages.append({"role": "user", "content": baseline_prompt})
    payload = json.dumps({
        "model": "gpt-4o",  
        "messages": messages,
        "max_tokens": len(encode_string_by_tiktoken(item["answer"])),
        "temperature": 1e-6,
        "top_p": 1.0,
        "logprobs": True,
        "top_logprobs": 10
    })
    headers = {
        'Authorization': f'Bearer {key}',
        'Content-Type': 'application/json'
    }
    response = requests.request("POST", url, headers=headers, data=payload)
    response = json.loads(response.text)
    choice = response['choices'][0]
    answer_token_strs = [decode_tokens_by_tiktoken([e]) for e in encode_string_by_tiktoken(item["answer"])]
    logprobs = choice['logprobs']['content']
    logit_list = []
    actual_token_logprobs = []
    # print(f"answer_token_strs: {answer_token_strs}")
    # print(f"logprobs: {logprobs}")
    for i, token_probs in enumerate(logprobs):
        token_data = []
        for entry in token_probs['top_logprobs']:
            token_data.append({
                "token": entry['token'],
                "logprob": entry['logprob'],
                "bytes": entry.get('bytes', None)
            })
        logit_list.append(token_data)

        current_token = answer_token_strs[i]
        match = next((e for e in token_data if e['token'] == current_token), None)
        if match:
            actual_token_logprobs.append(match['logprob'])
    avg_logprob = np.mean(actual_token_logprobs) if actual_token_logprobs else -20
    # print(f"avg_logprob: {avg_logprob}")
    return avg_logprob


base_logprob_list, element_logprob_list = [], []
for item, element in zip(dataset, elements_list):
    avg_logprob = get_avg_logprob(prompt_generator, item)
    elements = element["element"]
    element_list = []
    if elements:
        base_logprob_list.append(avg_logprob)
        for e in elements:
            element_list.append(get_avg_logprob(prompt_generator, item, e))
        print(f"element_list: {element_list}")
        element_logprob_list.append(max(element_list))

In [ ]:
logprobs = [sum(base_logprob_list) / len(base_logprob_list), sum(element_logprob_list) / len(element_logprob_list)]
print(f"base_logprob_list: {base_logprob_list}")
print(f"element_logprob_list: {element_logprob_list}")
print(f"logprobs: {logprobs}")

# calculate the CPR (experiment 4)

In [ ]:
ids = ""

with open(f"./results/{ids}/elements_list.json", "r") as f:
    filtered_elements_list = json.load(f)

with open(f"./results/{ids}/ablations/filtered_elements_wo_kg_list.json", "r") as f:
    filtered_elements_wo_kg_list = json.load(f)

with open(f"./results/{ids}/elements_list.json", "r") as f:
    elements_list = json.load(f)

import re

def answer_ratio_in_context(answer: str, context: str) -> float:
    answer = answer.strip()
    context = context.strip()
    pattern = r'\b' + re.escape(answer) + r'\b'
    matches = re.findall(pattern, context, flags=re.IGNORECASE)
    count = len(matches)
    answer_word_len = len(answer.split())
    context_word_len = len(context.split())

    if context_word_len == 0:
        return 0.0
    return (count * answer_word_len) / context_word_len


all_raw = all_kg = all_filter = all_full = 0

for item, x, y, z in zip(dataset, filtered_elements_list, filtered_elements_wo_kg_list, elements_list):
    count_kg = count_filter = count_full = 0
    id = item["id"]
    answer = item["answer"]
    raw_context = item["context"]
    wo_kg = y["element"]
    wo_filter = z["element"]
    full_method = x["element"]
    count_raw = answer_ratio_in_context(answer, raw_context)
    for kg in wo_kg:
        c = answer_ratio_in_context(answer, kg)
        count_kg += c
    count_kg = count_kg / len(wo_kg) if len(wo_kg) > 0 else 0
    for filter in wo_filter:
        c = answer_ratio_in_context(answer, filter)
        count_filter += c
    count_filter = count_filter / len(wo_filter) if len(wo_filter) > 0 else 0
    for full in full_method:
        c = answer_ratio_in_context(answer, full)
        count_full += c
    count_full = count_full / len(full_method) if len(full_method) > 0 else 0
    all_raw += count_raw
    all_kg += count_kg
    all_filter += count_filter
    all_full += count_full
    # print(f"id: {id}, answer: {answer}, count_raw: {count_raw}, count_kg: {count_kg}, count_filter: {count_filter}, count_full: {count_full}")

print(f"Raw: {100.0 * all_raw/len(dataset)}")
print(f"KG: {100.0 * all_kg/len(dataset)}")
print(f"Filter: {100.0 * all_filter/len(dataset)}")
print(f"Full: {100.0 * all_full/len(dataset)}")
